# SCBE-AETHERMOORE Fine-Tuning on Free Colab

## Train your own SCBE governance model using LoRA on a free T4 GPU

This notebook walks you through fine-tuning a small LLM (Llama-3.2-1B-Instruct) on SCBE governance training data using:

- **4-bit quantization** via bitsandbytes (fits in ~6GB VRAM)
- **LoRA** (Low-Rank Adaptation) so we only train a small fraction of parameters
- **SFTTrainer** from TRL for supervised fine-tuning

The training data comes from the SCBE-AETHERMOORE pipeline: governance-scored records ingested from Obsidian, Notion, browser swarm, spiral search, mesh events, and combat blockchain.

**Prerequisites:**
- A free Google Colab account with T4 GPU runtime
- A HuggingFace account with a write-access token
- The dataset already pushed to `issdandavis/scbe-aethermoore-training-data`

---

## Step 1: Install Dependencies

Install all required packages. This takes 2-3 minutes on a fresh Colab runtime.

- `transformers` — HuggingFace model loading and tokenization
- `peft` — Parameter-Efficient Fine-Tuning (LoRA)
- `bitsandbytes` — 4-bit quantization for memory efficiency
- `datasets` — HuggingFace dataset loading
- `trl` — SFTTrainer for supervised fine-tuning
- `huggingface_hub` — Authentication and model pushing
- `accelerate` — Distributed training utilities

In [ ]:
!pip install -q \
    transformers>=4.44.0 \
    peft>=0.12.0 \
    bitsandbytes>=0.43.0 \
    datasets>=2.20.0 \
    trl>=0.9.0 \
    huggingface_hub>=0.24.0 \
    accelerate>=0.33.0

print("All dependencies installed.")

# Verify GPU is available
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU detected: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

## Step 2: HuggingFace Login

Enter your HuggingFace token below. You need a **write-access** token to push the trained adapter back to HuggingFace.

Get your token at: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login, notebook_login

# This will display an interactive widget for you to paste your token
notebook_login()

print("HuggingFace login complete.")

## Step 3: Load the SCBE Training Dataset

Load the training data from `issdandavis/scbe-aethermoore-training-data` on HuggingFace.

The dataset is in JSONL format with the following schema:

```json
{
  "type": "web_research | obsidian | mesh_event | ...",
  "input": {
    "query": "the input prompt or question",
    "context": "additional context for the query"
  },
  "output": {
    "summary": "the response or answer",
    "governance_score": 0.0-1.0,
    "relevance_score": 0.0-1.0,
    "tongue_affinity": "KO|AV|RU|CA|UM|DR",
    "training_value": 0.0-1.0
  },
  "metadata": { ... }
}
```

In [ ]:
from datasets import load_dataset

DATASET_ID = "issdandavis/scbe-aethermoore-training-data"

# Load the dataset — try JSONL format first, then fall back to auto-detection
try:
    raw_dataset = load_dataset(DATASET_ID, split="train")
except Exception as e:
    print(f"Default load failed ({e}), trying explicit JSONL...")
    raw_dataset = load_dataset("json", data_files={"train": f"hf://datasets/{DATASET_ID}/*.jsonl"}, split="train")

print(f"Loaded {len(raw_dataset)} records from {DATASET_ID}")
print(f"\nColumns: {raw_dataset.column_names}")
print(f"\nSample record:")
print(raw_dataset[0])

## Step 4: Format Data for Supervised Fine-Tuning

We convert the structured SCBE records into a chat-style prompt/response format:

- **Prompt**: `input.query` (with optional `input.context`)
- **Response**: `output.summary` (enriched with governance metadata)

We also filter out low-quality records where `training_value <= 0.3` to ensure the model only learns from high-signal data.

In [ ]:
def format_for_sft(example):
    """Convert an SCBE record into a chat-formatted training example.

    Extracts the query from input and summary from output,
    then wraps them in the Llama-3 chat template format.
    """
    # Extract input fields — handle both dict and pre-flattened formats
    inp = example.get("input", {})
    out = example.get("output", {})

    if isinstance(inp, dict):
        query = inp.get("query", "")
        context = inp.get("context", "")
    else:
        query = str(inp)
        context = ""

    if isinstance(out, dict):
        summary = out.get("summary", "")
        gov_score = out.get("governance_score", 0.0)
        tongue = out.get("tongue_affinity", "unknown")
    else:
        summary = str(out)
        gov_score = 0.0
        tongue = "unknown"

    # Skip records with empty query or summary
    if not query or not summary:
        return {"text": ""}

    # Build the system message with SCBE governance context
    system_msg = (
        "You are an SCBE governance-aware AI assistant. "
        "You analyze queries through the SCBE 14-layer safety pipeline "
        "and respond with governance-scored, tongue-aligned outputs. "
        "Always consider data integrity, trust verification, and ethical alignment."
    )

    # Build the user message
    user_msg = query
    if context:
        user_msg += f"\n\nContext: {context}"

    # Build the assistant response with governance metadata
    assistant_msg = summary
    if gov_score > 0 or tongue != "unknown":
        assistant_msg += f"\n\n[Governance Score: {gov_score:.2f} | Tongue Affinity: {tongue}]"

    # Format as Llama-3 chat template
    text = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_msg}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_msg}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{assistant_msg}<|eot_id|>"
    )

    return {"text": text}


def get_training_value(example):
    """Extract training_value from the output field, handling nested dicts."""
    out = example.get("output", {})
    if isinstance(out, dict):
        return out.get("training_value", 0.0)
    return 0.0


# Filter by training_value > 0.3
print(f"Total records before filtering: {len(raw_dataset)}")
filtered_dataset = raw_dataset.filter(
    lambda x: get_training_value(x) > 0.3
)
print(f"Records after training_value > 0.3 filter: {len(filtered_dataset)}")

# Apply formatting
sft_dataset = filtered_dataset.map(format_for_sft, remove_columns=filtered_dataset.column_names)

# Remove any empty records
sft_dataset = sft_dataset.filter(lambda x: len(x["text"]) > 50)
print(f"Final SFT dataset size: {len(sft_dataset)}")

# Preview a formatted example
print("\n" + "="*80)
print("SAMPLE FORMATTED RECORD:")
print("="*80)
print(sft_dataset[0]["text"][:1000])

## Step 5: Load the Base Model with 4-bit Quantization

We use `unsloth/Llama-3.2-1B-Instruct` as the base model. At 1B parameters, this is small enough to fine-tune on a free T4 GPU (15GB VRAM) when loaded in 4-bit precision.

**4-bit quantization** (via bitsandbytes NF4) reduces the model's memory footprint from ~4GB (FP16) to ~0.7GB, leaving plenty of room for optimizer states, gradients, and activations.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "unsloth/Llama-3.2-1B-Instruct"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,  # nested quantization for extra savings
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Disable cache for training (saves memory)
model.config.use_cache = False

# Print model info
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {MODEL_ID}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters (before LoRA): {trainable_params:,}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 6: Configure LoRA Adapter

**LoRA** (Low-Rank Adaptation) freezes the base model weights and injects small trainable rank-decomposition matrices into the attention and MLP layers. This lets us fine-tune with:

- **rank=16**: The rank of the decomposition matrices (higher = more capacity, more memory)
- **alpha=32**: Scaling factor (alpha/rank = 2.0 effective learning rate multiplier)
- **Target modules**: All attention projections (q/k/v/o) and MLP layers (gate/up/down)

With this configuration, we train roughly 0.5-1% of the total parameters.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Prepare the model for k-bit training (handles gradient checkpointing, layer norms, etc.)
model = prepare_model_for_kbit_training(model)

# Define LoRA configuration
lora_config = LoraConfig(
    r=16,                      # LoRA rank
    lora_alpha=32,             # Scaling factor
    target_modules=[
        "q_proj",              # Query projection
        "k_proj",              # Key projection
        "v_proj",              # Value projection
        "o_proj",              # Output projection
        "gate_proj",           # MLP gate projection
        "up_proj",             # MLP up projection
        "down_proj",           # MLP down projection
    ],
    lora_dropout=0.05,         # Small dropout for regularization
    bias="none",               # Don't train bias terms
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable parameter stats
model.print_trainable_parameters()

print(f"\nGPU memory after LoRA setup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 7: Train with SFTTrainer

We use TRL's `SFTTrainer` for supervised fine-tuning. The training configuration is optimized for a free T4 GPU:

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| max_steps | 500 | ~1000 effective examples with grad accumulation |
| batch_size | 2 | Fits in T4 VRAM with 4-bit model |
| gradient_accumulation | 4 | Effective batch size of 8 |
| learning_rate | 2e-4 | Standard for LoRA fine-tuning |
| fp16 | True | T4 supports FP16 natively |
| max_seq_length | 512 | Keeps memory manageable |

Expected training time: **15-30 minutes** on a T4 GPU.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# Output directory for checkpoints
OUTPUT_DIR = "./scbe-lora-checkpoints"

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=500,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    save_steps=100,
    save_total_limit=2,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",      # 8-bit optimizer to save memory
    weight_decay=0.01,
    report_to="none",              # Disable wandb/tensorboard for simplicity
    seed=42,
)

# Create SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset,
    args=training_args,
    max_seq_length=512,
    dataset_text_field="text",
    packing=True,                  # Pack multiple short examples into one sequence
)

print(f"Training dataset: {len(sft_dataset)} examples")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Max steps: {training_args.max_steps}")
print(f"\nStarting training...\n")

# Train!
train_result = trainer.train()

# Print training summary
print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)
print(f"Total steps: {train_result.global_step}")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"GPU memory peak: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

## Step 8: Save and Push the LoRA Adapter to HuggingFace

We save only the LoRA adapter weights (not the full model). This is typically just 10-50MB, making it easy to share and deploy.

The adapter is pushed to `issdandavis/scbe-aethermoore-lora-v1` on HuggingFace. To use it later, you load the base model and apply the adapter on top.

In [ ]:
ADAPTER_REPO = "issdandavis/scbe-aethermoore-lora-v1"
LOCAL_ADAPTER_PATH = "./scbe-lora-final"

# Save the LoRA adapter locally
print("Saving LoRA adapter locally...")
trainer.model.save_pretrained(LOCAL_ADAPTER_PATH)
tokenizer.save_pretrained(LOCAL_ADAPTER_PATH)

# List saved files
import os
saved_files = os.listdir(LOCAL_ADAPTER_PATH)
total_size = sum(os.path.getsize(os.path.join(LOCAL_ADAPTER_PATH, f)) for f in saved_files)
print(f"Saved {len(saved_files)} files ({total_size / 1e6:.1f} MB):")
for f in sorted(saved_files):
    size = os.path.getsize(os.path.join(LOCAL_ADAPTER_PATH, f))
    print(f"  {f} ({size / 1e6:.1f} MB)")

# Push to HuggingFace
print(f"\nPushing adapter to {ADAPTER_REPO}...")
trainer.model.push_to_hub(
    ADAPTER_REPO,
    commit_message="SCBE-AETHERMOORE LoRA v1 — governance fine-tune on Llama-3.2-1B-Instruct",
)
tokenizer.push_to_hub(
    ADAPTER_REPO,
    commit_message="Add tokenizer for SCBE-AETHERMOORE LoRA v1",
)

print(f"\nAdapter pushed to: https://huggingface.co/{ADAPTER_REPO}")
print("Done!")

## Step 9: Test Inference

Let's test the fine-tuned model with a sample SCBE governance query. We reload the base model and apply the LoRA adapter to demonstrate the full inference pipeline.

This is the same process you would use in production: load the base model, apply the adapter, and generate.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Configuration
BASE_MODEL_ID = "unsloth/Llama-3.2-1B-Instruct"
ADAPTER_REPO = "issdandavis/scbe-aethermoore-lora-v1"

# Reload the base model in 4-bit
print("Loading base model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# Apply the LoRA adapter
print("Applying SCBE LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval()

print("Model ready for inference.\n")

# --- Test Queries ---

test_queries = [
    "Evaluate the governance compliance of an AI agent that autonomously scrapes user data from public forums without consent disclosure. What SCBE layers apply?",
    "How should the SCBE 14-layer pipeline handle a Sacred Egg mutation request that conflicts with the KO tongue's integrity constraints?",
    "Assess the trust score for a new data source claiming to provide verified blockchain-notarized training records.",
]

system_msg = (
    "You are an SCBE governance-aware AI assistant. "
    "You analyze queries through the SCBE 14-layer safety pipeline "
    "and respond with governance-scored, tongue-aligned outputs. "
    "Always consider data integrity, trust verification, and ethical alignment."
)

for i, query in enumerate(test_queries, 1):
    print(f"{'='*80}")
    print(f"QUERY {i}: {query}")
    print(f"{'='*80}")

    # Format as Llama-3 chat
    prompt = (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"{system_msg}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{query}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )

    # Decode only the generated tokens (skip the prompt)
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    print(f"\nRESPONSE:\n{response}\n")

print("="*80)
print("Inference test complete.")
print(f"\nYour fine-tuned adapter is available at:")
print(f"  https://huggingface.co/{ADAPTER_REPO}")
print(f"\nTo use in production:")
print(f"  from peft import PeftModel")
print(f"  model = PeftModel.from_pretrained(base_model, '{ADAPTER_REPO}')")